# 01 - Layout: chọn qubit vật lý nào làm qubit logic nào

Mạch bạn viết dùng **qubit logic** $q_0, q_1, \dots$ - chỉ là nhãn. Thiết bị có **qubit vật lý**
$Q_0, Q_1, \dots$ nối với nhau theo một đồ thị cố định, và chất lượng từng qubit khác nhau.

**Layout** (còn gọi là *initial mapping* hay *qubit allocation*) là phép gán
$\pi: \text{qubit logic} \to \text{qubit vật lý}$.

Đây là quyết định đầu tiên của compiler, và nó quan trọng hơn vẻ ngoài rất nhiều: notebook này sẽ
cho thấy **cùng một mạch, cùng một thiết bị**, chọn layout khác nhau thì số SWAP phải chèn là 3 hoặc
0. Layout tốt có thể làm bài toán routing biến mất hoàn toàn.

Notebook này cũng chỉ ra **hai cái bẫy** mà bất cứ ai tự dựng pipeline routing (tức là bất cứ ai
viết environment RL cho routing) cũng sẽ gặp - cả hai đều **im lặng**, không báo lỗi:

1. `SabreLayout` tự route bên trong, nối nó với `ApplyLayout` sẽ sinh mạch **không hợp lệ**.
2. `Operator(mạch_đã_compile).equiv(Operator(mạch_gốc))` trả về `False` **dù mạch đúng**.

In [ ]:
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit, transpile
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.quantum_info import Operator
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import (
    ApplyLayout,
    CheckMap,
    DenseLayout,
    EnlargeWithAncilla,
    FullAncillaAllocation,
    SabreLayout,
    SabreSwap,
    TrivialLayout,
    VF2Layout,
    VF2PostLayout,
)

print("Vườn coupling map:")
for ten, cm in [
    ("line(5)", CouplingMap.from_line(5)),
    ("ring(5)", CouplingMap.from_ring(5)),
    ("grid(2,4)", CouplingMap.from_grid(2, 4)),
    ("heavy_hex(3)", CouplingMap.from_heavy_hex(3)),
    ("full(5)", CouplingMap.from_full(5)),
]:
    n = cm.size()
    e = len(set(tuple(sorted(x)) for x in cm.get_edges()))
    print(f"  {ten:14s} {n:2d} qubit, {e:2d} edge, bậc trung bình {2*e/n:.1f}")

Số edge trên mỗi qubit ("bậc") là thước đo độ khó: `line` có bậc ~1.6 nên rất chật, `full` thì
routing không bao giờ cần thiết. Thiết bị siêu dẫn thật nằm ở phía chật - heavy-hex của IBM có bậc
chỉ khoảng 2.1.

**Distance matrix** là dữ liệu quan trọng nhất của một coupling map: $d(i,j)$ = số SWAP tối thiểu để
đưa hai qubit về kề nhau. Mọi heuristic routing đều tính cost từ ma trận này.

In [ ]:
COUPLING = CouplingMap.from_line(5)
print("distance matrix của line(5):")
print(pd.DataFrame(COUPLING.distance_matrix.astype(int),
                   index=[f"Q{i}" for i in range(5)],
                   columns=[f"Q{i}" for i in range(5)]).to_string())
print("\n-> cx(Q0, Q4) cần d(0,4) - 1 = 3 lần SWAP trước khi thực hiện được.")

## 1. Bài toán layout = tìm subgraph isomorphism

Mỗi mạch có một **interaction graph**: đỉnh là qubit logic, cạnh là "có ít nhất một cổng 2-qubit
giữa cặp này". Nếu interaction graph **nhúng được** vào coupling map (tồn tại phép gán sao cho mọi
cạnh cần đến đều có sẵn), thì tồn tại layout cần **0 SWAP**.

Tìm phép nhúng đó chính là bài toán **subgraph isomorphism** - NP-hard. Và khi không nhúng được
(mạch dày hơn thiết bị), bài toán trở thành tối thiểu hoá số SWAP, cũng NP-hard.

In [ ]:
mach = QuantumCircuit(4)
mach.h(0)
mach.cx(0, 3)
mach.cx(1, 3)
mach.cx(0, 2)

CM4 = CouplingMap.from_line(4)


def interaction_graph(qc):
    """Tập cặp qubit logic có cổng 2-qubit, kèm số lần tương tác."""
    dem = {}
    for ins in qc.data:
        if len(ins.qubits) == 2:
            cap = tuple(sorted(qc.find_bit(q).index for q in ins.qubits))
            dem[cap] = dem.get(cap, 0) + 1
    return dem


ig = interaction_graph(mach)
co_san = set(tuple(sorted(e)) for e in CM4.get_edges())

print("interaction graph của mạch:", ig)
print("edge có sẵn trên line(4) :", sorted(co_san))
print()
for cap, n in sorted(ig.items()):
    print(f"  cần {cap} ({n} cổng): {'có sẵn' if cap in co_san else 'KHÔNG kề nhau -> cần SWAP'}")
print("\n-> với layout tầm thường (q_i -> Q_i) thì CẢ BA cặp đều không kề nhau.")

## 2. Bốn pass layout của Qiskit

| Pass | Cách làm | Khi nào dùng |
|---|---|---|
| `TrivialLayout` | $q_i \to Q_i$, không suy nghĩ gì | làm baseline để so |
| `DenseLayout` | chọn vùng con **dày nhất** (nhiều edge nhất) của thiết bị | nhanh, không nhìn mạch |
| `VF2Layout` | subgraph isomorphism **chính xác** (thuật toán VF2++) | tìm được layout 0 SWAP nếu có |
| `SabreLayout` | heuristic SABRE, chạy layout **và** routing cùng lúc, nhiều trial | mặc định của Qiskit |

Để chạy một layout pass riêng rồi tự route, công thức chuẩn là năm pass nối nhau:

```python
PassManager([
    <LayoutPass>,                 # chọn layout, ghi vào property_set["layout"]
    FullAncillaAllocation(cm),    # thêm qubit vật lý còn lại làm ancilla
    EnlargeWithAncilla(),         # mở rộng mạch cho đủ số qubit vật lý
    ApplyLayout(),                # ĐỔI TÊN qubit logic sang qubit vật lý
    <RoutingPass>,                # chèn SWAP
])
```

Công thức này dùng lại ở notebook `02`, `04` và `05`, nên nhớ nó.

In [ ]:
def chay_pipeline(layout_pass, qc, cm, router=None, seed=0):
    """Layout + routing riêng, trả về mạch đã route và property_set."""
    router = router if router is not None else SabreSwap(cm, heuristic="decay", seed=seed)
    pm = PassManager([
        layout_pass,
        FullAncillaAllocation(cm),
        EnlargeWithAncilla(),
        ApplyLayout(),
        router,
        CheckMap(cm),          # kiểm tra: mọi cổng 2-qubit có hợp lệ không
    ])
    return pm.run(qc), pm.property_set


print(f"{'layout pass':16s} {'swap':>5s} {'depth':>6s} {'hợp lệ':>7s}  layout (logic -> vật lý)")
for ten, lp in [
    ("TrivialLayout", TrivialLayout(CM4)),
    ("DenseLayout", DenseLayout(CM4)),
    ("VF2Layout", VF2Layout(CM4, seed=0)),
]:
    out, ps = chay_pipeline(lp, mach, CM4)
    lay = ps["layout"]
    pi = [lay[q] for q in mach.qubits]
    print(f"  {ten:14s} {out.count_ops().get('swap', 0):5d} {out.depth():6d} "
          f"{str(ps['is_swap_mapped']):>7s}  {pi}")

print("\n-> VF2Layout tìm được phép nhúng hoàn hảo: 0 SWAP.")
print("-> TrivialLayout và DenseLayout phải trả giá 3 SWAP = 9 cổng cx cho CÙNG mạch đó.")

Đó là toàn bộ giá trị của layout: **3 SWAP so với 0 SWAP**, chỉ khác nhau ở chỗ đánh số qubit. 3
SWAP là 9 cổng `cx` thêm vào một mạch vốn chỉ có 3 cổng `cx`.

## 3. Bẫy thứ nhất: `SabreLayout` tự route bên trong

`SabreLayout` **không chỉ** chọn layout. Nó chạy cả thuật toán SABRE routing để đánh giá layout, và
trả về mạch **đã được apply layout và đã route**. Điều này không viết ở tên pass.

Hệ quả: nếu bạn nối `SabreLayout` với `ApplyLayout` theo công thức ở trên, layout bị **áp dụng hai
lần**. Mạch kết quả không hợp lệ, và **không có exception nào được raise** - chỉ `CheckMap` phát
hiện được.

In [ ]:
def hai_qubit(qc):
    return [(i.operation.name, tuple(qc.find_bit(q).index for q in i.qubits))
            for i in qc.data if len(i.qubits) == 2]


print("A) SabreLayout một mình (nó tự lo cả layout lẫn routing):")
pm = PassManager([SabreLayout(CM4, seed=0), CheckMap(CM4)])
out_a = pm.run(mach)
print("   cổng 2-qubit:", hai_qubit(out_a))
print("   is_swap_mapped:", pm.property_set["is_swap_mapped"], "<- hợp lệ")

print("\nB) SabreLayout + ApplyLayout (apply layout LẦN THỨ HAI):")
pm = PassManager([SabreLayout(CM4, seed=0), FullAncillaAllocation(CM4),
                  EnlargeWithAncilla(), ApplyLayout(), CheckMap(CM4)])
out_b = pm.run(mach)
print("   cổng 2-qubit:", hai_qubit(out_b))
print("   is_swap_mapped:", pm.property_set["is_swap_mapped"], "<- HỎNG, mà không báo lỗi gì")
print("   cx(3,0) và cx(2,0) không phải edge của line(4) -> thiết bị không chạy được mạch này.")

**Quy tắc rút ra:** dùng layout pass *chỉ chọn layout* (`TrivialLayout`, `DenseLayout`, `VF2Layout`)
khi bạn muốn tự route. `SabreLayout` chỉ dùng một mình. Dấu hiệu nhận biết: `SabreLayout` ghi
`property_set["final_layout"]` (bằng chứng nó đã route), ba pass kia thì không.

**Và luôn đặt `CheckMap` ở cuối mọi pipeline tự dựng.** Nó đọc `property_set["is_swap_mapped"]`.
Ở notebook `05`, đây là cổng kiểm tra bắt buộc cho output của RL agent - không có nó thì một agent
sinh mạch sai vẫn "thắng" baseline vì nó bỏ qua các cổng khó.

In [ ]:
for ten, lp in [("TrivialLayout", TrivialLayout(CM4)), ("VF2Layout", VF2Layout(CM4, seed=0)),
                ("DenseLayout", DenseLayout(CM4)), ("SabreLayout", SabreLayout(CM4, seed=0))]:
    pm = PassManager([lp])
    pm.run(mach)
    co_route = pm.property_set["final_layout"] is not None
    print(f"  {ten:14s} final_layout {'CÓ  -> pass này đã route, dùng một mình' if co_route else 'không -> an toàn để nối ApplyLayout'}")

## 4. Ba phép hoán vị của `TranspileLayout`

Sau khi `transpile`, kết quả mang theo `qc.layout` thuộc lớp `TranspileLayout`. Nó chứa **ba** thông
tin khác nhau, rất dễ lẫn:

| Phương thức | Nghĩa |
|---|---|
| `initial_index_layout()` | layout **ban đầu**: qubit logic $i$ được đặt vào qubit vật lý nào |
| `routing_permutation()` | hoán vị do **các SWAP** gây ra trong lúc routing |
| `final_index_layout()` | kết hợp cả hai: cuối mạch, qubit logic $i$ đang nằm ở qubit vật lý nào |

Nếu mạch không cần SWAP thì `routing_permutation()` là phép đồng nhất và `final` trùng `initial`.

In [ ]:
compiled = transpile(mach, coupling_map=CM4, basis_gates=["cx", "h", "rz", "sx", "x"],
                     optimization_level=1, seed_transpiler=3)
L = compiled.layout

print("initial_index_layout():", L.initial_index_layout())
print("routing_permutation() :", L.routing_permutation())
print("final_index_layout()  :", L.final_index_layout())
print("\nsố swap trong mạch:", compiled.count_ops().get("swap", 0))
print("-> routing_permutation là đồng nhất vì transpile tìm được layout không cần SWAP.")

## 5. Bẫy thứ hai: kiểm tra tương đương sau khi compile

Đây là cái bẫy làm mất nhiều thời gian nhất cho người tự viết compiler hoặc RL agent.

Cách kiểm tra "mạch compile ra có đúng không" tưởng như hiển nhiên:

```python
Operator(compiled).equiv(Operator(goc))      # -> False, DÙ MẠCH ĐÚNG
```

Nó trả về `False` vì `Operator(compiled)` đọc mạch theo **qubit vật lý**, còn `Operator(goc)` theo
**qubit logic**. Layout đã hoán vị chúng, nên hai unitary khác nhau đúng bằng một phép hoán vị.

`Operator.from_circuit()` biết đọc `qc.layout` và tự bù lại hoán vị đó:

In [ ]:
U_goc = Operator(mach)

print("Operator(compiled).equiv(Operator(mạch gốc))           =",
      Operator(compiled).equiv(U_goc), "  <- SAI, bẫy ở đây")
print("Operator.from_circuit(compiled).equiv(Operator(gốc))   =",
      Operator.from_circuit(compiled).equiv(U_goc), "  <- đúng")

# Và nó cũng đúng với mạch có SWAP thật: mạch route bằng TrivialLayout cần 3 SWAP
out_t, _ = chay_pipeline(TrivialLayout(CM4), mach, CM4)
print("\nmạch route bằng TrivialLayout, có", out_t.count_ops().get("swap", 0), "SWAP:")
print("  Operator(out).equiv(gốc)              =", Operator(out_t).equiv(U_goc))
print("  Operator.from_circuit(out).equiv(gốc) =", Operator.from_circuit(out_t).equiv(U_goc))
print("\n-> Khi tự viết router, SWAP ở cuối mạch có thể bỏ được (chỉ đổi nhãn qubit),")
print("   nhưng chỉ khi bạn theo dõi hoán vị. Dùng from_circuit để không tự lừa mình.")

## 6. Noise-aware layout: không phải qubit nào cũng như nhau

Tới giờ ta coi mọi edge của coupling map là bình đẳng. Trên thiết bị thật thì không: error rate giữa
edge tốt nhất và tệ nhất chênh nhau vài lần. Nên có một cách giảm sai số **không cần giảm cổng nào**:
đặt mạch lên vùng qubit tốt.

`VF2PostLayout` làm đúng việc đó - sau khi mạch đã route xong, nó tìm một phép nhúng khác cùng hình
dạng nhưng trên các edge có error rate thấp hơn, dựa vào `Target` của backend.

In [ ]:
backend = GenericBackendV2(num_qubits=5, coupling_map=CouplingMap.from_line(5).get_edges(), seed=11)
loi_cx = {e: p.error for e, p in backend.target["cx"].items()}

mach3 = QuantumCircuit(3)
mach3.cx(0, 1)
mach3.cx(1, 2)
mach3.cx(0, 1)


def xac_suat_khong_loi(qc):
    """Tích (1 - error) trên từng cổng cx, theo đúng edge vật lý nó nằm."""
    p = 1.0
    for ins in qc.data:
        if ins.operation.name == "cx":
            a, b = (qc.find_bit(q).index for q in ins.qubits)
            p *= 1 - loi_cx[(a, b)]
    return p


t1 = transpile(mach3, backend=backend, optimization_level=1,
               seed_transpiler=5, initial_layout=[0, 1, 2])
edges1 = [tuple(t1.find_bit(q).index for q in i.qubits) for i in t1.data if i.operation.name == "cx"]
print(f"layout [0,1,2]: cx trên edge {edges1}")
print(f"                error từng edge {[round(loi_cx[e], 5) for e in edges1]}")
print(f"                P(không lỗi) = {xac_suat_khong_loi(t1):.5f}")

pm = PassManager([VF2PostLayout(target=backend.target, seed=5)])
pm.run(t1)
post = pm.property_set["post_layout"]
tot_hon = [post[q] for q in t1.qubits][:3]

t2 = transpile(mach3, backend=backend, optimization_level=1,
               seed_transpiler=5, initial_layout=tot_hon)
edges2 = [tuple(t2.find_bit(q).index for q in i.qubits) for i in t2.data if i.operation.name == "cx"]
print(f"\nVF2PostLayout đề xuất layout {tot_hon}:")
print(f"                cx trên edge {edges2}")
print(f"                error từng edge {[round(loi_cx[e], 5) for e in edges2]}")
print(f"                P(không lỗi) = {xac_suat_khong_loi(t2):.5f}")

e1, e2 = 1 - xac_suat_khong_loi(t1), 1 - xac_suat_khong_loi(t2)
print(f"\n-> cùng số cổng, cùng hình dạng, nhưng sai số giảm {(1 - e2/e1) * 100:.0f}%.")

Hàm `xac_suat_khong_loi` ở trên là một **hàm cost noise-aware**. Giữ nó trong đầu: ở notebook `05`
nó sẽ là một trong ba lựa chọn reward cho RL agent, và là lựa chọn gần với mục tiêu thật nhất -
vì cái ta muốn cuối cùng không phải "ít SWAP" mà là "kết quả đo được đúng".

## Tóm tắt

| Khái niệm | Nội dung |
|---|---|
| layout | phép gán $\pi$: qubit logic → qubit vật lý |
| interaction graph | cặp qubit logic nào cần cổng 2-qubit |
| layout hoàn hảo | interaction graph nhúng được vào coupling map → 0 SWAP |
| `TrivialLayout` / `DenseLayout` / `VF2Layout` | chỉ chọn layout, an toàn để nối `ApplyLayout` |
| `SabreLayout` | **tự route luôn** - dùng một mình, ghi `final_layout` |
| công thức tự route | layout → `FullAncillaAllocation` → `EnlargeWithAncilla` → `ApplyLayout` → router → `CheckMap` |
| `CheckMap` | ghi `is_swap_mapped`, cổng kiểm tra bắt buộc |
| `TranspileLayout` | `initial_index_layout` / `routing_permutation` / `final_index_layout` |
| kiểm tra tương đương | `Operator.from_circuit(out)`, **không** phải `Operator(out)` |
| noise-aware | `VF2PostLayout` + error rate từ `Target` |

### Thử tự làm

1. Đổi `CM4` thành `CouplingMap.from_ring(4)`. `TrivialLayout` còn cần SWAP nữa không? Vì sao?
2. Tạo một mạch có interaction graph là **ngôi sao** (`cx(0,1)`, `cx(0,2)`, `cx(0,3)`) rồi thử
   nhúng vào `line(4)`. `VF2Layout` có tìm được layout 0 SWAP không? (Gợi ý: đỉnh bậc 3 cần một
   qubit vật lý có 3 edge - `line` không có.)
3. Chạy `VF2Layout` trên mạch 5 qubit ngẫu nhiên (`random_circuit(5, 20, max_operands=2)`).
   Kiểm tra `property_set["VF2Layout_stop_reason"]` - vì sao nó không tìm được layout hoàn hảo?
4. Viết lại `xac_suat_khong_loi` để tính cả error của cổng 1-qubit
   (`backend.target["sx"][(q,)].error`). Cổng 1-qubit đóng góp bao nhiêu phần vào tổng sai số?